# HGP clusterer sur MarieBenchmark

Ce notebook :

- Installe les dépendances système et Python nécessaires,
- Récupère **hgp_clusterer** depuis GitHub,
- Lance **HypergraphPercol** localement sur MarieBenchmark,
- Propose une visualisation 3D des résultats.

# Installation de HypergraphPercol (Universal Setup)

In [1]:
# @title Configuration du Backend
# Choix du backend géométrique : 'geogram' (Rapide, Recommandé) ou 'cgal' (Historique, Exact)
BACKEND = 'geogram'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")


Backend sélectionné : geogram


In [2]:
import os
import subprocess

def run_sh(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

print("Installation des outils de build...")
run_sh('apt-get update -qq')
# libeigen3-dev est requis pour HGP (kernels_geogram.hpp)
run_sh('apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev')

if BACKEND == 'cgal':
    print("Installation des dépendances CGAL/TBB...")
    run_sh('apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev libboost-all-dev')
elif BACKEND == 'geogram':
    print("Backend Geogram: Pas de dépendances système supplémentaires (Build headless).")


Installation des outils de build...
$ apt-get update -qq
$ apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev
Backend Geogram: Pas de dépendances système supplémentaires (Build headless).


In [3]:
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.7 MB/s eta 0:00:00


In [4]:
%%bash
set -euo pipefail
WORKDIR="${HGP_WORKDIR:-/content}"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    echo "Mise à jour de HGP-clusterer (Force Reset)..."
    cd HGP-clusterer
    git fetch origin
    git reset --hard origin/main
    cd ..
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

# Cyminiball
if [ -d cyminiball ]; then
    echo "Mise à jour de cyminiball..."
    cd cyminiball
    git pull --ff-only
    cd ..
else
    git clone https://github.com/Ludwig-H/cyminiball.git
fi


Cloning into 'HGP-clusterer'...
remote: Internal Server Error
fatal: unable to access 'https://github.com/Ludwig-H/HGP-clusterer.git/': The requested URL returned error: 500


CalledProcessError: Command 'b'set -euo pipefail\nWORKDIR="${HGP_WORKDIR:-/content}"\nmkdir -p "${WORKDIR}"\ncd "${WORKDIR}"\n\n# HGP-clusterer\nif [ -d HGP-clusterer ]; then\n    echo "Mise \xc3\xa0 jour de HGP-clusterer (Force Reset)..."\n    cd HGP-clusterer\n    git fetch origin\n    git reset --hard origin/main\n    cd ..\nelse\n    git clone https://github.com/Ludwig-H/HGP-clusterer.git\nfi\n\n# Cyminiball\nif [ -d cyminiball ]; then\n    echo "Mise \xc3\xa0 jour de cyminiball..."\n    cd cyminiball\n    git pull --ff-only\n    cd ..\nelse\n    git clone https://github.com/Ludwig-H/cyminiball.git\nfi\n'' returned non-zero exit status 128.

In [ ]:
%%bash
set -euo pipefail
WORKDIR="${HGP_WORKDIR:-/content}"
mkdir -p "${WORKDIR}/wheels"
cd "${WORKDIR}/cyminiball"
python3 -m pip wheel --no-build-isolation --no-deps --wheel-dir="${WORKDIR}/wheels" .
python3 -m pip install --force-reinstall --no-deps --no-index --find-links="${WORKDIR}/wheels" cyminiball


In [ ]:
import os
import sys
from pathlib import Path

WORKDIR = os.environ.get('HGP_WORKDIR', '/content')
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        get_ipython().system('git clone --recursive https://github.com/BrunoLevy/geogram.git')

    print("Compilation de Geogram (Headless)...")
    # Utilisation directe de CMake pour désactiver les graphismes (évite les dépendances X11)
    get_ipython().system('cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF')
    get_ipython().system('cmake --build geogram/build --config Release --parallel 4')
    get_ipython().system('cmake --install geogram/build --prefix /usr/local')

    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'
    print("Geogram installé.")

elif BACKEND == 'cgal':
    print("Configuration des dépendances CGAL locales...")
    setup_script = f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"
    get_ipython().system(f'python3 {setup_script}')

    cgal_dir = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"
    projects = [
        'EdgesCGALDelaunay2D', 'EdgesCGALDelaunay3D', 'EdgesCGALDelaunayND',
        'EdgesCGALWeightedDelaunay2D', 'EdgesCGALWeightedDelaunay3D', 'EdgesCGALWeightedDelaunayND'
    ]
    for proj in projects:
        p_path = f"{cgal_dir}/{proj}"
        get_ipython().system(f'cmake -S {p_path} -B {p_path}/build -DCMAKE_BUILD_TYPE=Release')
        get_ipython().system(f'cmake --build {p_path}/build --config Release')
        get_ipython().system(f'cmake --install {p_path}/build --prefix {WORKDIR}/HGP-clusterer')


In [ ]:
%%bash
set -euo pipefail
WORKDIR="${HGP_WORKDIR:-/content}"
cd "${WORKDIR}/HGP-clusterer"

# Force Geogram detection
export GEOGRAM_INSTALL_PREFIX=/usr/local

echo "Nettoyage des anciens builds..."
rm -rf build dist *.egg-info

echo "Installation de HGP-clusterer (Backend détecté par CMake)..."
python3 -m pip install -v --no-deps .


In [ ]:
import os

workdir = os.environ.get("HGP_WORKDIR", "/content")
repo_root = os.path.join(workdir, "HGP-clusterer")
os.environ["CGALDELAUNAY_ROOT"] = os.path.join(repo_root, "CGALDelaunay")
# os.environ['CGAL_NTHREADS'] = '4' # Uncomment to limit threads manually. Default: All available threads.\n
from hgp_clusterer import HGPClusterer

## Paramètres / imports

In [ ]:
# Cellule 2 : imports, paramètres globaux, wrappers

import numpy as np
import pandas as pd

import hdbscan
import plotly.express as px

from IPython.display import display

import plotly.io as pio
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

# Active le mode offline pour Colab
init_notebook_mode(connected=True)


# Pour que Plotly s'affiche correctement dans Colab
pio.renderers.default = "colab"


# -------------------------------------------------------------------
# Import / wrapper HGPClusterer
# -------------------------------------------------------------------

try:
    # Adapte si le module est ailleurs
    from hgp_clusterer import HGPClusterer
except ImportError:
    HGPClusterer = None
    print("⚠️ HGPClusterer introuvable. Adapte l'import ci-dessus.")

In [ ]:
# Visualisations 3D robustes avec go.Scatter3d + iplot
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import plotly.graph_objects as go
from plotly.offline import iplot

def plot_3d_single(X, labels, title, max_points=100000, plot_anomalies=False):
    """
    Affiche un nuage 3D coloré par 'labels' avec go.Scatter3d + iplot,
    en utilisant une palette étendue (jusqu'à 60 couleurs distinctes, puis spectrale).

    Args:
        plot_anomalies (bool): Si True, regroupe tous les clusters >= 0 sous le label 0.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels).reshape(-1)

    if X.ndim != 2 or X.shape[1] < 3:
        raise ValueError(f"X doit être de shape (n_samples, >=3), reçu {X.shape}")

    if len(X) != len(labels):
        raise ValueError(f"Taille incompatible: len(X)={len(X)}, len(labels)={len(labels)}")

    # Downsampling si trop de points
    n = len(X)
    if n > max_points:
        idx = np.random.choice(n, size=max_points, replace=False)
        X = X[idx]
        labels = labels[idx]

    # Si mode anomalies, on copie pour ne pas modifier l'original (si pas downsamplé) ou la copie (si downsamplé)
    # et on remplace tous les clusters valides par 0
    if plot_anomalies:
        labels = labels.copy()
        labels[labels >= 0] = 0

    unique_labels = np.unique(labels)
    # On exclut le bruit (-1) et le hors champ (-2) pour la palette
    clusters = unique_labels[(unique_labels != -1) & (unique_labels != -2)]
    n_clusters = len(clusters)

    # --- Génération Palette ---
    palette = []
    if n_clusters > 0:
        if n_clusters <= 20:
            # Jusqu'à 20 couleurs : tab20
            cmap = plt.get_cmap("tab20")
            palette = [mcolors.to_hex(cmap(i)) for i in range(n_clusters)]
        elif n_clusters <= 40:
            # Jusqu'à 40 couleurs : tab20 + tab20b
            c1 = [mcolors.to_hex(plt.get_cmap("tab20")(i)) for i in range(20)]
            c2 = [mcolors.to_hex(plt.get_cmap("tab20b")(i)) for i in range(n_clusters - 20)]
            palette = c1 + c2
        elif n_clusters <= 60:
            # Jusqu'à 60 couleurs : tab20 + tab20b + tab20c
            c1 = [mcolors.to_hex(plt.get_cmap("tab20")(i)) for i in range(20)]
            c2 = [mcolors.to_hex(plt.get_cmap("tab20b")(i)) for i in range(20)]
            c3 = [mcolors.to_hex(plt.get_cmap("tab20c")(i)) for i in range(n_clusters - 40)]
            palette = c1 + c2 + c3
        else:
            # Plus de 60 : nipy_spectral (très contrasté)
            cmap = plt.get_cmap("nipy_spectral")
            palette = [mcolors.to_hex(cmap(i/n_clusters)) for i in range(n_clusters)]
            # Mélange pour séparer les couleurs proches pour des clusters proches
            np.random.RandomState(42).shuffle(palette)

    cluster_color_map = {c: palette[i] for i, c in enumerate(clusters)}

    fig = go.Figure()

    for lab in unique_labels:
        mask = labels == lab
        if not np.any(mask):
            continue

        if lab == -2:
            # Hors champ : noir, presque transparent
            lab_name = f"Cluster {lab} (Out of Field)"
            marker_opts = dict(
                size=2,
                opacity=0.05,
                color='black'
            )
        elif lab == -1:
            # Bruit : gris, transparent
            lab_name = f"Cluster {lab} (Noise)"
            marker_opts = dict(
                size=2,
                opacity=0.1,
                color='lightgray'
            )
        else:
            # Si mode anomalie, le cluster 0 représente tous les objets valides
            if plot_anomalies and lab == 0:
                lab_name = "Valid Objects (Merged)"
            elif lab <= -3:
                 lab_name = f"Anomaly {lab}"
            else:
                lab_name = f"Cluster {lab}"

            col = cluster_color_map[lab]
            marker_opts = dict(
                size=3,
                opacity=0.4,
                color=col
            )

        fig.add_trace(go.Scatter3d(
            x=X[mask, 0].tolist(),
            y=X[mask, 1].tolist(),
            z=X[mask, 2].tolist(),
            mode='markers',
            name=lab_name,
            marker=marker_opts
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        showlegend=True
    )

    iplot(fig)

## Résultat sur MarieBenchmark

In [ ]:
# Installation de plyfile et téléchargement du fichier
!pip install -q plyfile
!gdown 1k3Bd_HucRn1v066Xed5P_JRY8pPbiXIS -O scan_lidar.ply

In [ ]:
from plyfile import PlyData
import numpy as np

# Chargement du fichier PLY
plydata = PlyData.read('scan_lidar.ply')
vertex_data = plydata['vertex']

x = vertex_data['x']
y = vertex_data['y']
z = vertex_data['z']
X_ply = np.column_stack([x, y, z])

print(f"Nuage de points chargé : {X_ply.shape[0]} points.")

## Download Bounding Box Files and set ground truth vector



In [ ]:
!pip install --upgrade gdown
!gdown --folder "https://drive.google.com/drive/folders/1oCfegLY2e8uv4Wcs1kFTijrB1vD7BAcE?usp=sharing" -O bounding_boxes
!ls bounding_boxes

In [ ]:
import os
import glob
import numpy as np

def parse_off_bounds(filepath):
    """
    Parses a simple OFF file to extract the bounding box min and max coordinates.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Filter empty lines and comments
    lines = [line.strip() for line in lines if line.strip() and not line.strip().startswith('#')]

    if not lines:
        return None, None

    # Handle Header
    header = lines[0]
    start_line = 0

    n_verts = 0

    if header.upper() == 'OFF':
        # Counts are on the next line
        try:
            n_verts = int(lines[1].split()[0])
            start_line = 2
        except IndexError:
            return None, None
    elif header.upper().startswith('OFF'):
        # Counts are on the same line, e.g., OFF 8 6 0
        parts = header[3:].split()
        if len(parts) > 0:
            n_verts = int(parts[0])
            start_line = 1
        else:
             # Fallback if just 'OFF' but split weirdly
             try:
                n_verts = int(lines[1].split()[0])
                start_line = 2
             except IndexError:
                return None, None
    else:
        # No OFF header, assume first line has counts
        try:
            n_verts = int(header.split()[0])
            start_line = 1
        except ValueError:
            return None, None

    vertices = []
    for i in range(n_verts):
        if start_line + i >= len(lines):
            break
        parts = lines[start_line + i].split()
        # Take first 3 as x, y, z (ignore color if present)
        coords = [float(p) for p in parts[:3]]
        vertices.append(coords)

    if not vertices:
        return None, None

    vertices_np = np.array(vertices)
    min_bounds = np.min(vertices_np, axis=0)
    max_bounds = np.max(vertices_np, axis=0)

    return min_bounds, max_bounds

# Retrieve and sort files
off_files = sorted(glob.glob('bounding_boxes/*.off'))
bb_data = []

print(f"Found {len(off_files)} .off files.")

for idx, fpath in enumerate(off_files):
    min_b, max_b = parse_off_bounds(fpath)
    if min_b is not None:
        fname = os.path.basename(fpath)
        bb_info = {
            'id': idx,
            'filename': fname,
            'min': min_b,
            'max': max_b
        }
        bb_data.append(bb_info)
        print(f"ID: {idx} | File: {fname}")
        print(f"  Min: {min_b}")
        print(f"  Max: {max_b}")
    else:
        print(f"Could not parse {fpath}")

print(f"\nSuccessfully parsed {len(bb_data)} bounding boxes.")

In [ ]:
import numpy as np

# 1. Calcul de la Bounding Box géante (globale) englobant toutes les boîtes
margin = 0.2

if bb_data:
    all_mins = np.array([bb['min'] for bb in bb_data])
    all_maxs = np.array([bb['max'] for bb in bb_data])
    # Le min global est le min des mins, le max global est le max des maxs
    # On applique la marge : on recule le min et on avance le max
    global_min = np.min(all_mins, axis=0) - margin
    global_max = np.max(all_maxs, axis=0) + margin
else:
    # Fallback (peu probable ici)
    global_min = np.min(X_ply, axis=0)
    global_max = np.max(X_ply, axis=0)

print(f"Global Bounding Box calculated from {len(bb_data)} boxes (margin={margin}):")
print(f"  Min: {global_min}")
print(f"  Max: {global_max}")

# 2. Initialisation du vecteur ground_truth
# Par défaut, tout est du bruit (-1)
ground_truth = np.full(X_ply.shape[0], -1, dtype=int)

# 3. Marquage des points "hors champ" (-2)
# Un point est hors champ s'il est en dehors de [global_min, global_max]
mask_inside_global = np.all((X_ply >= global_min) & (X_ply <= global_max), axis=1)
ground_truth[~mask_inside_global] = -2

print(f"Points outside global bbox (Cluster -2): {np.sum(~mask_inside_global)}")

# 4. Assignation des clusters spécifiques
for bb in bb_data:
    b_min = bb['min']
    b_max = bb['max']
    cid = bb['id']

    # Points inside this specific box
    mask = np.all((X_ply >= b_min) & (X_ply <= b_max), axis=1)

    # Assign the ID (this overwrites -1 or -2 if applicable)
    ground_truth[mask] = cid

# Statistics
unique_labels, counts = np.unique(ground_truth, return_counts=True)
print(f"\nGround Truth Summary:")
for label, count in zip(unique_labels, counts):
    if label == -1:
        print(f"  Noise (-1): {count}")
    elif label == -2:
        print(f"  Out of Field (-2): {count}")
    else:
        print(f"  Cluster {label}: {count}")

In [ ]:
plot_3d_single(X_ply, ground_truth, "Ground Truth : Clusters et Hors-champ (-2)")

## Paramètres HGP-clusterer

In [ ]:
# -------------------------------------------------------------------
# Paramètres globaux
# -------------------------------------------------------------------

# Paramètres pour HypergraphPercol & HDBSCAN
K = 3
subsample = 1.0
min_cluster_size = int(200 * subsample)
min_samples = K + 1
method = 'eom'

weight_face = "lambda"      # "lambda" ∝ 1/r ; "uniform" ∝ 1 ; "unique" 1 sur la face arg_min r
label_all_points = False
return_multi_clusters = False
complex_chosen = "orderk_delaunay"
expZ = 2
cgal_root = "/content/HGP-clusterer/CGALDelaunay"
verbeux = True


In [ ]:
# Paramètres globaux pour la fonction de splitting
τ = 0.30
δτ = 0.05

def splitting_τ(cluster_père, clusters_fils):
    """
    Fonction de décision de splitting basée sur la pureté par rapport au ground_truth.

    Args:
        cluster_père: np.array d'index des points du cluster père.
        clusters_fils: liste de np.arrays d'index des clusters fils.

    Returns:
        True si on préfère les fils (split), False si on garde le père (merge).
    """
    # On utilise le vecteur ground_truth global
    global ground_truth

    # --- 1. Calcul pour le père ---
    labels_pere = ground_truth[cluster_père]

    if len(labels_pere) == 0:
        return False

    # Proportion du label majoritaire dans le père
    vals_pere, counts_pere = np.unique(labels_pere, return_counts=True)
    p_pere = counts_pere.max() / len(labels_pere)

    # --- 2. Calcul pour les fils (moyenne pondérée) ---
    weighted_purity_sum = 0
    total_fils_size = 0

    for fils_indices in clusters_fils:
        n_fils = len(fils_indices)
        if n_fils == 0:
            continue

        labels_fils = ground_truth[fils_indices]
        vals_fils, counts_fils = np.unique(labels_fils, return_counts=True)

        # Proportion du label majoritaire dans ce fils
        p_local = counts_fils.max() / n_fils

        weighted_purity_sum += p_local * n_fils
        total_fils_size += n_fils

    if total_fils_size == 0:
        return False

    p_fils = weighted_purity_sum / total_fils_size

    # --- 3. Décision ---
    if p_fils < τ:
        return False
    elif p_fils > p_pere + δτ:
        return True
    else:
        return False

## HGP-clusterer

In [ ]:
# Exécution de HGPClusterer sur le nuage de points
print("Lancement de HGPClusterer sur les données PLY...")

splitting = None # splitting_τ

# On réutilise les paramètres globaux définis plus haut (K, min_cluster_size, etc.)
clusterer = HGPClusterer(
    K=K,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    method=method,
    splitting=splitting,
    weight_face=weight_face,
    label_all_points=label_all_points,
    return_multi_clusters=return_multi_clusters,
    complex_chosen=complex_chosen,
    expZ=expZ,
    subsample=subsample,
    backend=BACKEND,
    cgal_root=cgal_root,
    verbose=verbeux
)

labels_ply = clusterer.fit_predict(X_ply)

print("Clustering terminé.")

In [ ]:
# Visualisation
plot_3d_single(X_ply, labels_ply, "Résultat HGPClusterer sur scan_lidar.ply")

In [ ]:
import numpy as np

print("Calcul de labels_anomalies...")

# Initialisation : copie de labels_ply
labels_anomalies = labels_ply.copy()

# On récupère la liste des clusters trouvés par HGP (hors bruit -1)
unique_clusters = np.unique(labels_ply)
unique_clusters = unique_clusters[unique_clusters != -1]

current_anomaly_id = -3

# --- 1ère Passe : Classification des clusters ---
for c in unique_clusters:
    # Indices des points du cluster courant
    mask_c = (labels_ply == c)

    # Labels ground_truth correspondants
    gt_values = ground_truth[mask_c]

    if len(gt_values) == 0:
        continue

    # Compte des labels et tri décroissant
    vals, counts = np.unique(gt_values, return_counts=True)
    sorted_idx = np.argsort(-counts)
    vals_sorted = vals[sorted_idx]
    counts_sorted = counts[sorted_idx]
    total_points = len(gt_values)

    decision_label = None

    # Boucle sur les candidats (du plus fréquent au moins fréquent)
    for v, count in zip(vals_sorted, counts_sorted):
        prop = count / total_points

        if v == -2:
            # Si majoritaire est -2, il faut > 70%
            if prop > 0.7:
                decision_label = -2
                break
            else:
                continue # On passe au suivant

        elif v == -1:
            # "Même règle pour le calcul des anomalies" (si -1, il faut > 50%)
            if prop > 0.5:
                decision_label = -1
                break
            else:
                continue # On passe au suivant
        else:
            # Label valide (>= 0) : on le prend direct s'il arrive en tête (ou après rejet des autres)
            decision_label = v
            break

    # Application de la décision
    if decision_label == -2:
        labels_anomalies[mask_c] = -2
    elif decision_label == -1 or decision_label is None:
        # Si c'est du bruit confirmé ou si on n'a rien trouvé (cas pathologique mélange bruit/hors champ < 50%)
        labels_anomalies[mask_c] = current_anomaly_id
        current_anomaly_id -= 1
    else:
        # decision_label >= 0 : C'est un objet valide.
        # On garde le label original de labels_ply.
        pass

# Identification des anomalies créées
unique_anomalies = np.unique(labels_anomalies)
unique_anomalies = unique_anomalies[unique_anomalies <= -3]

print(f"Passe 1 terminée. {len(unique_anomalies)} clusters d'anomalies identifiés.")

# --- 2ème Passe : Absorption du bruit dans les Bounding Boxes des anomalies ---
count_absorbed = 0

for anom_id in unique_anomalies:
    # Points appartenant à l'anomalie en cours
    mask_anom = (labels_anomalies == anom_id)
    X_anom = X_ply[mask_anom]

    if len(X_anom) == 0: continue

    # Calcul de la Bounding Box de l'anomalie
    min_b = np.min(X_anom, axis=0)
    max_b = np.max(X_anom, axis=0)

    # Masque des points qui sont encore du bruit (-1) ET qui sont du bruit dans la vérité terrain
    mask_noise = (labels_anomalies == -1) & (ground_truth == -1)

    # Masque des points dans la Bounding Box
    mask_in_box = np.all((X_ply >= min_b) & (X_ply <= max_b), axis=1)

    # Intersection : Bruit ET dans la Boîte
    mask_to_update = mask_noise & mask_in_box

    # Mise à jour
    n_updated = np.sum(mask_to_update)
    if n_updated > 0:
        labels_anomalies[mask_to_update] = anom_id
        count_absorbed += n_updated

print(f"Passe 2 terminée. {count_absorbed} points de bruit absorbés dans les anomalies.")

# Résumé final
unique_final = np.unique(labels_anomalies)
n_anom = np.sum(unique_final <= -3)
n_noise = np.sum(labels_anomalies == -1)
n_out = np.sum(labels_anomalies == -2)
n_valid = len(unique_final) - (1 if -1 in unique_final else 0) - (1 if -2 in unique_final else 0) - n_anom

print(f"\nRésumé labels_anomalies :")
print(f"  Clusters Valides (>=0) : {n_valid}")
print(f"  Clusters Anomalies (<= -3) : {n_anom}")
print(f"  Bruit restant (-1) : {n_noise}")
print(f"  Hors champ (-2) : {n_out}")

In [ ]:
plot_3d_single(X_ply, labels_anomalies, "Résultat : Anomalies (<= -3) vs Objets Valides (0)", plot_anomalies=True)

In [ ]:
import numpy as np

print("Post-processing : Fusion des anomalies intersectantes...")

labels_definitive_anomalies = labels_anomalies.copy()
unique_anoms = np.unique(labels_definitive_anomalies)
unique_anoms = unique_anoms[unique_anoms <= -3]

if len(unique_anoms) > 0:
    # 1. Calcul des BBox pour chaque anomalie actuelle
    anom_bboxes = {}
    for uid in unique_anoms:
        mask = (labels_definitive_anomalies == uid)
        pts = X_ply[mask]
        anom_bboxes[uid] = {
            'min': np.min(pts, axis=0),
            'max': np.max(pts, axis=0)
        }

    # 2. Construction du graphe d'adjacence (intersection)
    n_anoms = len(unique_anoms)
    adj_matrix = np.eye(n_anoms, dtype=bool)

    def boxes_intersect(b1, b2):
        # Check AABB intersection : max(mins) <= min(maxs)
        inter_min = np.maximum(b1['min'], b2['min'])
        inter_max = np.minimum(b1['max'], b2['max'])
        return np.all(inter_min <= inter_max)

    for i in range(n_anoms):
        for j in range(i + 1, n_anoms):
            u1 = unique_anoms[i]
            u2 = unique_anoms[j]
            if boxes_intersect(anom_bboxes[u1], anom_bboxes[u2]):
                adj_matrix[i, j] = True
                adj_matrix[j, i] = True

    # 3. Recherche des composantes connexes (BFS)
    visited = np.zeros(n_anoms, dtype=bool)
    new_mapping = {} # old_label -> new_label
    current_new_label = -3

    for i in range(n_anoms):
        if not visited[i]:
            # Start BFS pour cette composante
            stack = [i]
            visited[i] = True
            component_indices = []

            while stack:
                curr = stack.pop()
                component_indices.append(curr)
                # Voisins
                neighbors = np.where(adj_matrix[curr])[0]
                for nbr in neighbors:
                    if not visited[nbr]:
                        visited[nbr] = True
                        stack.append(nbr)

            # Assigne le nouveau label à toute la composante
            for idx in component_indices:
                old_lab = unique_anoms[idx]
                new_mapping[old_lab] = current_new_label

            current_new_label -= 1

    # 4. Application du re-mapping
    # On utilise un masque pour chaque ancien label pour mettre à jour vers le nouveau
    temp_labels = labels_definitive_anomalies.copy()
    for old_l, new_l in new_mapping.items():
        mask = (labels_anomalies == old_l)
        temp_labels[mask] = new_l

    labels_definitive_anomalies = temp_labels

    n_merged_anoms = abs(current_new_label - (-3))
    print(f"Fusion terminée. {len(unique_anoms)} anomalies initiales fusionnées en {n_merged_anoms} anomalies définitives.")

else:
    print("Aucune anomalie à fusionner.")

# --- 2ème Passe : Absorption du bruit (sur les nouvelles anomalies) ---
print("Absorption du bruit dans les anomalies fusionnées...")

unique_final_anoms = np.unique(labels_definitive_anomalies)
unique_final_anoms = unique_final_anoms[unique_final_anoms <= -3]
count_absorbed = 0

for anom_id in unique_final_anoms:
    # Recalcul de la BBox fusionnée
    mask_anom = (labels_definitive_anomalies == anom_id)
    X_anom = X_ply[mask_anom]

    if len(X_anom) == 0: continue

    min_b = np.min(X_anom, axis=0)
    max_b = np.max(X_anom, axis=0)

    # Points bruités (-1) qui tombent dans cette bbox ET sont du bruit dans la vérité terrain
    mask_noise = (labels_definitive_anomalies == -1) & (ground_truth == -1)
    mask_in_box = np.all((X_ply >= min_b) & (X_ply <= max_b), axis=1)

    mask_update = mask_noise & mask_in_box
    n_up = np.sum(mask_update)

    if n_up > 0:
        labels_definitive_anomalies[mask_update] = anom_id
        count_absorbed += n_up

print(f"Post-processing terminé. {count_absorbed} points de bruit supplémentaires absorbés.")

# Stats finales
unique, counts = np.unique(labels_definitive_anomalies, return_counts=True)
print("\nRépartition finale (top 10) :")
for l, c in zip(unique[:10], counts[:10]):
    print(f"  Cluster {l}: {c}")

In [ ]:
plot_3d_single(X_ply, labels_definitive_anomalies, "Résultat Définitif : Anomalies Fusionnées", plot_anomalies=True)

In [ ]:
# Exécution de HDBSCAN sur le nuage de points (comparaison)
print("Lancement de HDBSCAN sur les données PLY...")
import time

t0 = time.time()
# Utilisation des mêmes paramètres globaux définis précédemment
clusterer_hdb = hdbscan.HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    cluster_selection_method=method
)
labels_hdb_ply = clusterer_hdb.fit_predict(X_ply)

n_clusters_hdb_ply = len(set(labels_hdb_ply)) - (1 if -1 in labels_hdb_ply else 0)
print(f"HDBSCAN terminé en {time.time()-t0:.2f}s. Clusters trouvés : {n_clusters_hdb_ply}")

In [ ]:
# Visualisation HDBSCAN
plot_3d_single(X_ply, labels_hdb_ply, "Résultat HDBSCAN sur scan_lidar.ply")

## Exemple « jouet », pour vérifier que tout fonctionne !

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import make_blobs, make_moons
import hdbscan

# Import de votre librairie
from hgp_clusterer import HGPClusterer

# --- 1. Génération du Dataset "Advanced HDBSCAN style" ---
# On génère assez de points (10k) pour que le subsample=0.1 soit pertinent
n_samples = 10000
moons, _ = make_moons(n_samples=n_samples // 2, noise=0.05)
blobs, _ = make_blobs(n_samples=n_samples // 2, centers=[(-0.75, 2.25), (1.0, 2.0), (-2, 0.5)], cluster_std=[0.3, 0.25, 0.4])
# On décale les moons pour qu'elles se mélangent bien
moons[:, 0] -= 0.5
X = np.vstack([moons, blobs])
# Ajout de bruit uniforme
rng = np.random.default_rng(42)
noise = rng.uniform(low=np.min(X, axis=0)-1, high=np.max(X, axis=0)+1, size=(int(n_samples * 0.1), 2))
X = np.vstack([X, noise])

print(f"Jeu de données généré : {X.shape[0]} points")

# --- 2. Paramètres ---
K = 15
min_cluster_size = 100
min_samples = K+1
method = 'eom'
splitting = None
weight_face = "lambda"      # "lambda" ∝ 1/r ; "uniform" ∝ 1 ; "unique" 1 sur la face min r
label_all_points = False
return_multi_clusters = False
complex_chosen = "orderk_delaunay"
expZ = 2
subsample = 1.0
cgal_root = "/content/HGP-clusterer/CGALDelaunay"
verbeux = True

# --- 3. Exécution HGPClusterer ---
print("\n--- Lancement HGPClusterer ---")
start_time = time.time()

clusterer_hgp = HGPClusterer(
    K=K,
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    metric="euclidean",
    method=method,
    splitting=splitting,
    weight_face=weight_face,
    label_all_points=label_all_points,
    return_multi_clusters=return_multi_clusters,
    complex_chosen=complex_chosen,
    expZ=expZ,
    subsample=subsample,
    backend=BACKEND,
    cgal_root=cgal_root,
    verbose=verbeux
)

labels_hgp = clusterer_hgp.fit_predict(X)

hgp_time = time.time() - start_time
n_clusters_hgp = len(np.unique(labels_hgp)) - (1 if -1 in labels_hgp else 0)
print(f"HGPClusterer terminé en {hgp_time:.2f}s. Clusters trouvés : {n_clusters_hgp}")

# --- 4. Exécution HDBSCAN (Baseline) ---
print("\n--- Lancement HDBSCAN Standard ---")
start_time = time.time()
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples,
    cluster_selection_method=method
)
labels_hdb = clusterer.fit_predict(X)
hdb_time = time.time() - start_time
n_clusters_hdb = len(np.unique(labels_hdb)) - (1 if -1 in labels_hdb else 0)
print(f"HDBSCAN terminé en {hdb_time:.2f}s. Clusters trouvés : {n_clusters_hdb}")

# --- 5. Comparaison Visuelle ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Palette de couleurs : gris pour le bruit (-1), couleurs vives pour le reste
def plot_clusters(data, labels, ax, title):
    unique_labels = np.unique(labels)
    colors = plt.cm.turbo(np.linspace(0, 1, len(unique_labels)))

    # On trace le bruit en premier et en gris
    noise_mask = labels == -1
    ax.scatter(data[noise_mask, 0], data[noise_mask, 1], c='lightgray', s=2, alpha=0.3, label='Noise')

    # On trace les clusters
    for k, col in zip(unique_labels, colors):
        if k == -1: continue
        mask = labels == k
        ax.scatter(data[mask, 0], data[mask, 1], color=col, s=5, alpha=0.8)

    ax.set_title(title, fontsize=14)
    ax.axis('off')

plot_clusters(X, labels_hdb, axes[0], f"HDBSCAN\nTime: {hdb_time:.2f}s | Clusters: {n_clusters_hdb}")
plot_clusters(X, labels_hgp, axes[1], f"HGPClusterer (Subsample={subsample})\nTime: {hgp_time:.2f}s | Clusters: {n_clusters_hgp}")

plt.tight_layout()
plt.show()